# Which pool should the SVC train on? Measured, not assumed.

Our SVC carries weight 12 of roughly 28 total — the largest member — and `_load_pool()` picks its training
table by **row count** (`if len(df) <= best_n: continue`), never by provenance. The candidates differ
enormously:

* **A `illiadolenko/ntl-merged-corpus-v2`** — a merge of many sources, no dedup, no label audit. This is what
  the current notebook uses, because it is the largest.
* **B `crtproduction/name-that-llm-open-pool`** — built to be clean: `flag_train_prompt_regen` marks passages
  regenerated from train prompts, and `gemini37_unverified.parquet` holds rows whose routing is suspect.
* **C `gtuli12/ntl-own-pool`** — 39 datasets plus open pool v12, exact and 5-gram near-duplicates removed,
  30-300 words, no markdown, with a `train_prompt` flag.

A published forensic analysis reports that cleaning the public pool was worth **+0.027 accuracy** to the
models retrained on it — the largest single lever anyone has reported for this competition. We have never
tested it.

This trains the **same 4-view SVC** on each pool and reports prompt-grouped OOF at **k=18 and k=2**. The LB
tracks k=2 (r=0.91) rather than k=18 (r=0.13), so the k=2 column is the one that decides.

In [ ]:
"""Which pool should the SVC train on? Measure it instead of guessing.

Our SVC carries weight 12 of roughly 28 total — the single largest member — and it trains on whichever
passage table `_load_pool()` finds largest. That function compares `len(df)`, so it picks by row count and
never looks at quality. The candidates on Kaggle differ enormously in provenance:

  * `illiadolenko/ntl-merged-corpus-v2`  — a merge of many sources, no dedup, no label audit
  * `crtproduction/name-that-llm-open-pool` — built to be clean: `flag_train_prompt_regen` marks passages
    regenerated from train prompts, and `gemini37_unverified.parquet` holds rows whose routing is suspect
  * `gtuli12/ntl-own-pool` — 39 datasets plus open pool v12, with exact and 5-gram near-duplicates removed,
    30-300 words, no markdown, and a `train_prompt` flag

A published forensic analysis of this competition reports that cleaning the public pool was worth +0.027
accuracy to the models retrained on it. That is the largest single lever anyone has reported, and we have
never tested it on our own SVC.

This notebook trains the same 4-view SVC on each pool in turn and reports prompt-grouped OOF at both k=18
and k=2 (the LB tracks k=2, r=0.91). Same views, same folds, same official rows — only the pool changes.
"""
import glob
import json
import os
import re
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

T0 = time.time()
INPUT_DIR = "/kaggle/input"
DATA = next(p for p in [f"{INPUT_DIR}/competitions/name-that-llm", f"{INPUT_DIR}/name-that-llm"]
            if os.path.exists(f"{p}/train.csv"))
train = pd.read_csv(f"{DATA}/train.csv")
LABELS = sorted(train.model.unique().tolist())
LAB2I = {m: i for i, m in enumerate(LABELS)}
K = len(LABELS)
y = train.model.map(LAB2I).to_numpy()
groups = train.prompt_id.astype(str).to_numpy()
print(f"official {len(train):,} rows, {len(set(groups))} prompts, {K} labels", flush=True)

SMALL = set("a an the and or but if while although because as so than that which who whom whose what when "
            "where why how of in on at to for with from by about into over under between during before "
            "after above below up down out off again further then once here there all any both each few "
            "more most other some such no nor not only own same too very can will just should now is are "
            "was were be been being have has had do does did".split())


def shape_text(t):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    return " ".join(w.lower() if w.lower() in SMALL or not w.isalnum() else "CONTENT" for w in t.split())


VIEWS = [("char", dict(analyzer="char_wb", ngram_range=(3, 5), max_features=250000), 1.0),
         ("word", dict(analyzer="word", ngram_range=(1, 3), max_features=180000), 0.7),
         ("shape", dict(analyzer="char_wb", ngram_range=(2, 6), max_features=60000), 0.5),
         ("small", dict(analyzer="word", ngram_range=(1, 3), max_features=50000), 0.5)]
print(f"{len(VIEWS)} views", flush=True)

## Build each pool with the cleaning its own author documents

In [ ]:
"""Build each candidate pool with the cleaning its own author documents, and report what changed."""
def _find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    return hits[0] if hits else None


def _finish(df, name):
    """Filters every candidate gets, so the comparison is about provenance, not preprocessing."""
    before = len(df)
    df = df.dropna(subset=["text", "model"]).copy()
    df["text"] = df["text"].astype(str)
    df["model"] = df["model"].astype(str)
    df = df[df.model.isin(LABELS)]
    df = df[df.text.str.split().str.len().between(30, 300)]
    # exact duplicates of an official train passage are leakage, not data
    train_norm = set(train.text.astype(str).str.replace(r"\s+", " ", regex=True).str.strip())
    df["_n"] = df.text.str.replace(r"\s+", " ", regex=True).str.strip()
    leaked = df._n.isin(train_norm).sum()
    df = df[~df._n.isin(train_norm)]
    df = df.drop_duplicates(subset=["_n"]).drop(columns=["_n"])
    print(f"  {name:<22} {before:>8,} -> {len(df):>8,} rows | dropped {leaked:>5,} train-duplicates", flush=True)
    return df


pools = {}

p = _find("merged_corpus.csv")
if p:
    d = pd.read_csv(p)
    cols = {c.lower(): c for c in d.columns}
    t, m = cols.get("text"), cols.get("model") or cols.get("label")
    pools["A merged (ours now)"] = _finish(d[[t, m]].rename(columns={t: "text", m: "model"}), "A merged")

p = _find("ntl_open_pool.parquet")
if p:
    d = pd.read_parquet(p)
    print(f"  open pool columns: {list(d.columns)[:12]}", flush=True)
    if "flag_train_prompt_regen" in d.columns:
        n0 = len(d)
        d = d[~d.flag_train_prompt_regen.astype(bool)]
        print(f"  open pool: dropped {n0-len(d):,} rows regenerated from train prompts", flush=True)
    d = d[["text", "model"]]
    bad = _find("gemini37_unverified.parquet")
    pools["B open pool (clean)"] = _finish(d, "B open pool")

p = _find("pool_own_v3.parquet")
if p:
    d = pd.read_parquet(p)
    print(f"  gtuli12 columns: {list(d.columns)[:14]}", flush=True)
    if "train_prompt" in d.columns:
        n0 = len(d)
        d = d[~d.train_prompt.astype(bool)]
        print(f"  gtuli12: dropped {n0-len(d):,} rows that answer a train prompt", flush=True)
    pools["C gtuli12 (cleanest)"] = _finish(d[["text", "model"]], "C gtuli12")

print(f"\n{len(pools)} candidate pools built", flush=True)
for name, d in pools.items():
    vc = d.model.value_counts()
    print(f"  {name:<22} {len(d):>8,} rows | per class {vc.min():>5,}-{vc.max():>5,} | "
          f"classes present {len(vc)}", flush=True)

## Same SVC, same folds, same official rows — only the pool changes

In [ ]:
"""Same 4-view SVC on each pool, prompt-grouped OOF at k=18 and k=2."""
def k2_subsample(groups_arr, per_prompt=2, seed=17):
    rng = np.random.default_rng(seed)
    keep = []
    for g in np.unique(groups_arr):
        idx = np.flatnonzero(groups_arr == g)
        if len(idx) > per_prompt:
            idx = rng.choice(idx, size=per_prompt, replace=False)
        keep.append(idx)
    return np.concatenate(keep)


def balanced(pred, true):
    per = [(pred[true == c] == c).mean() for c in range(K) if (true == c).any()]
    return float(np.mean(per))


results = {}
k2_rows = k2_subsample(groups)

for name, pool in pools.items():
    t_pool = time.time()
    Xp, yp = pool.text.tolist(), pool.model.map(LAB2I).to_numpy()
    oof = np.zeros((len(train), K), np.float32)
    for fi, (itr, iva) in enumerate(GroupKFold(2).split(np.arange(len(train)), y, groups), 1):
        # official fold-train rows + the whole pool; the pool is fixed auxiliary data in every fold
        fit_texts = train.text.astype(str).values[itr].tolist() + Xp
        fit_y = np.r_[y[itr], yp]
        score = np.zeros((len(iva), K), np.float32)
        for vname, kwargs, w in VIEWS:
            prep = shape_text if vname == "shape" else small_words_text if vname == "small" else (lambda s: s)
            texts = [prep(t) for t in fit_texts]
            vec = TfidfVectorizer(sublinear_tf=True, **kwargs)
            Xf = vec.fit_transform(texts)
            clf = LinearSVC(C=0.5, class_weight="balanced", max_iter=3000)
            clf.fit(Xf, fit_y)
            Xv = vec.transform([prep(t) for t in train.text.astype(str).values[iva]])
            d = clf.decision_function(Xv)
            score += w * d
        oof[iva] = score
    ba18 = balanced(oof.argmax(1), y)
    ba2 = balanced(oof[k2_rows].argmax(1), y[k2_rows])
    results[name] = {"rows": int(len(pool)), "oof_k18": round(ba18, 4), "oof_k2": round(ba2, 4),
                     "seconds": round(time.time() - t_pool)}
    print(f"{name:<24} k18 {ba18:.4f} | k2 {ba2:.4f} | {time.time()-t_pool:.0f}s", flush=True)

print("\n=== RESULT ===", flush=True)
best = max(results.items(), key=lambda kv: kv[1]["oof_k2"])
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["oof_k2"]):
    print(f"  {name:<24} k2 {r['oof_k2']:.4f}  ({r['rows']:,} rows)", flush=True)
print(f"\nBEST on the metric the LB tracks (k=2): {best[0]}", flush=True)
json.dump(results, open("/kaggle/working/pool_comparison.json", "w"), indent=2)
print("wrote pool_comparison.json", flush=True)